In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

from sklearn.metrics import (
    roc_curve,
    roc_auc_score,
    precision_recall_curve,
    average_precision_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)

# ==========================================
# 1. Setup
# ==========================================

os.makedirs(
    "outputs/evaluation",
    exist_ok=True
)

# ==========================================
# 2. Load Data
# ==========================================

pred = pd.read_csv(
    "data/final_test_predictions.csv",
    parse_dates=["Date"],
    index_col="Date"
)

market = pd.read_csv(
    "data/market_data_final_features.csv",
    parse_dates=["Date"],
    index_col="Date"
)

shap_imp = pd.read_csv(
    "data/shap_feature_importance.csv"
)

# Align
df = pred.join(
    market[
        [
            "SP500",
            "VIX",
            "Future_Drawdown_10D"
        ]
    ],
    how="left"
)

y_true = df["Actual"].values
y_prob = df["Crash_Probability"].values

BEST_THRESHOLD = 0.62
WARNING_THRESHOLD = 0.37

y_pred = (
    y_prob >= BEST_THRESHOLD
).astype(int)

roc_auc = roc_auc_score(
    y_true,
    y_prob
)

pr_auc = average_precision_score(
    y_true,
    y_prob
)

# ==========================================
# 3. ROC Curve
# ==========================================

fpr, tpr, _ = roc_curve(
    y_true,
    y_prob
)

plt.figure(figsize=(8, 6))

plt.plot(
    fpr,
    tpr,
    label=f"ROC-AUC = {roc_auc:.3f}"
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--"
)

plt.xlabel(
    "False Positive Rate"
)

plt.ylabel(
    "True Positive Rate"
)

plt.title(
    "ROC Curve – Market + TDA Model"
)

plt.legend()

plt.grid(
    alpha=0.3
)

plt.tight_layout()

plt.savefig(
    "outputs/evaluation/roc_curve.png",
    dpi=300
)

plt.close()

# ==========================================
# 4. Precision-Recall Curve
# ==========================================

precision, recall, _ = (
    precision_recall_curve(
        y_true,
        y_prob
    )
)

baseline = np.mean(
    y_true
)

plt.figure(
    figsize=(8, 6)
)

plt.plot(
    recall,
    precision,
    label=f"PR-AUC = {pr_auc:.3f}"
)

plt.axhline(
    baseline,
    linestyle="--",
    label=f"Random baseline = {baseline:.3f}"
)

plt.xlabel(
    "Recall"
)

plt.ylabel(
    "Precision"
)

plt.title(
    "Precision-Recall Curve"
)

plt.legend()

plt.grid(
    alpha=0.3
)

plt.tight_layout()

plt.savefig(
    "outputs/evaluation/precision_recall_curve.png",
    dpi=300
)

plt.close()

# ==========================================
# 5. Confusion Matrix
# ==========================================

cm = confusion_matrix(
    y_true,
    y_pred
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=[
        "No Crash",
        "Crash Risk"
    ]
)

fig, ax = plt.subplots(
    figsize=(7, 6)
)

disp.plot(
    ax=ax,
    values_format="d"
)

plt.title(
    "Confusion Matrix – Threshold 0.62"
)

plt.tight_layout()

plt.savefig(
    "outputs/evaluation/confusion_matrix.png",
    dpi=300
)

plt.close()

# ==========================================
# 6. Top SHAP Features
# ==========================================

top = (
    shap_imp
    .head(15)
    .sort_values(
        "Mean_Abs_SHAP"
    )
)

plt.figure(
    figsize=(10, 7)
)

plt.barh(
    top["Feature"],
    top["Mean_Abs_SHAP"]
)

plt.xlabel(
    "Mean |SHAP Value|"
)

plt.title(
    "Top 15 Predictive Features"
)

plt.tight_layout()

plt.savefig(
    "outputs/evaluation/top_shap_features.png",
    dpi=300
)

plt.close()

# ==========================================
# 7. TDA Feature Importance
# ==========================================

tda = shap_imp[
    shap_imp["Feature"]
    .str.startswith("TDA_")
].copy()

tda = tda.sort_values(
    "Mean_Abs_SHAP"
)

plt.figure(
    figsize=(10, 6)
)

plt.barh(
    tda["Feature"],
    tda["Mean_Abs_SHAP"]
)

plt.xlabel(
    "Mean |SHAP Value|"
)

plt.title(
    "TDA Feature Importance"
)

plt.tight_layout()

plt.savefig(
    "outputs/evaluation/tda_feature_importance.png",
    dpi=300
)

plt.close()

# ==========================================
# 8. Crash Probability Timeline
# ==========================================

plt.figure(
    figsize=(15, 7)
)

plt.plot(
    df.index,
    df["Crash_Probability"],
    label="Crash Probability"
)

plt.axhline(
    WARNING_THRESHOLD,
    linestyle="--",
    label="Warning Threshold (0.37)"
)

plt.axhline(
    BEST_THRESHOLD,
    linestyle="--",
    label="High Risk Threshold (0.62)"
)

# Highlight actual crash-risk dates
crash_dates = df[
    df["Actual"] == 1
]

plt.scatter(
    crash_dates.index,
    crash_dates[
        "Crash_Probability"
    ],
    marker="x",
    label="Actual Crash-Risk Days"
)

plt.ylabel(
    "Predicted Crash Probability"
)

plt.xlabel(
    "Date"
)

plt.title(
    "Market Crash Risk Timeline"
)

plt.legend()

plt.grid(
    alpha=0.3
)

plt.tight_layout()

plt.savefig(
    "outputs/evaluation/crash_risk_timeline.png",
    dpi=300
)

plt.close()

# ==========================================
# 9. S&P 500 + Risk Signals
# ==========================================

plt.figure(
    figsize=(15, 7)
)

plt.plot(
    df.index,
    df["SP500"],
    label="S&P 500"
)

warning = df[
    (
        df["Crash_Probability"]
        >= WARNING_THRESHOLD
    )
    &
    (
        df["Crash_Probability"]
        < BEST_THRESHOLD
    )
]

high = df[
    df["Crash_Probability"]
    >= BEST_THRESHOLD
]

plt.scatter(
    warning.index,
    warning["SP500"],
    marker="o",
    s=20,
    label="Warning"
)

plt.scatter(
    high.index,
    high["SP500"],
    marker="x",
    s=35,
    label="High Risk"
)

plt.xlabel(
    "Date"
)

plt.ylabel(
    "S&P 500"
)

plt.title(
    "S&P 500 with TDA-Based Crash Risk Signals"
)

plt.legend()

plt.grid(
    alpha=0.3
)

plt.tight_layout()

plt.savefig(
    "outputs/evaluation/sp500_risk_signals.png",
    dpi=300
)

plt.close()

# ==========================================
# 10. Save Final Metrics
# ==========================================

tn, fp, fn, tp = cm.ravel()

precision_final = (
    tp / (tp + fp)
)

recall_final = (
    tp / (tp + fn)
)

f1_final = (
    2 * precision_final * recall_final
    / (precision_final + recall_final)
)

metrics = pd.DataFrame({

    "Metric": [
        "ROC-AUC",
        "PR-AUC",
        "Precision @ 0.62",
        "Recall @ 0.62",
        "F1 @ 0.62",
        "TDA SHAP Contribution"
    ],

    "Value": [
        roc_auc,
        pr_auc,
        precision_final,
        recall_final,
        f1_final,
        0.3812
    ]

})

metrics.to_csv(
    "data/final_metrics.csv",
    index=False
)

print(
    "\n========== FINAL METRICS =========="
)

print(
    metrics.to_string(
        index=False
    )
)

print(
    "\nVisualizations saved in:"
    " outputs/evaluation/"
)

print(
    "\nFiles:"
)

for file in os.listdir(
    "outputs/evaluation"
):
    print(file)


========== FINAL METRICS ==========
               Metric    Value
              ROC-AUC 0.836538
               PR-AUC 0.154652
     Precision @ 0.62 0.206522
        Recall @ 0.62 0.463415
            F1 @ 0.62 0.285714
TDA SHAP Contribution 0.381200

Visualizations saved in: outputs/evaluation/

Files:
confusion_matrix.png
crash_risk_timeline.png
precision_recall_curve.png
roc_curve.png
sp500_risk_signals.png
tda_feature_importance.png
top_shap_features.png
